# C05_E01 — Ajuste FOPDT por curve_fit (utilidad reutilizable)

**Capítulo:** 5 — Respuesta temporal, estabilidad y desempeño  
**Identificador:** `C05_E01`  
**Objetivo pedagógico:** Identificación canónica FOPDT como bloque reutilizable.  
**Librerías:** numpy, scipy.optimize, matplotlib

## Ejemplo industrial

Identificación de un horno tras escalón en consigna del SCADA.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. Función auxiliar reutilizable

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import os
np.random.seed(0)

def fopdt(t, K, tau, theta):
    """Respuesta a escalón unitario de un FOPDT (K, tau, theta)."""
    y = np.zeros_like(t, dtype=float)
    mask = t > theta
    y[mask] = K*(1.0 - np.exp(-(t[mask] - theta)/tau))
    return y

def fit_fopdt(t, y, p0=(1.0, 10.0, 1.0)):
    """Ajuste FOPDT a partir de t, y; devuelve (K, tau, theta)."""
    popt, _ = curve_fit(fopdt, t, y, p0=p0)
    return popt

## 2. Aplicación a datos sintéticos del Capítulo 5

In [2]:
t = np.linspace(0, 100, 200)
y_real = fopdt(t, K=2.0, tau=15.0, theta=5.0) + 0.05*np.random.randn(t.size)
K_h, tau_h, theta_h = fit_fopdt(t, y_real)
print(f"K={K_h:.3f}, tau={tau_h:.3f}, theta={theta_h:.3f}")

K=2.008, tau=15.432, theta=4.809


## 3. Figura

In [3]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, y_real, '.', alpha=0.6, label="Datos")
ax.plot(t, fopdt(t, K_h, tau_h, theta_h), '-', label="FOPDT ajustado")
ax.set_xlabel("Tiempo (s)"); ax.set_ylabel("Salida"); ax.grid(alpha=0.3); ax.legend()
ax.set_title("C05_E01 — FOPDT identificado (reutilizable)")
out = '../../figures/cap05/fig_C05_F01_fopdt_identificado.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()

## 4. Interpretación

El notebook expone `fopdt(...)` y `fit_fopdt(...)` como utilidades reutilizables en C10 (sintonía) y C13 (control adaptativo). La identificación con datos sintéticos recupera los parámetros con error compatible con el ruido inyectado.